# Modelagem e Inferência - VAREJO_05

Avaliação Prática Multidisciplinar: Infraestrutura de Big Data + Paradigmas e Tecnologias Emergentes.

Pipeline: S3 (raw) → Glue ETL → S3 (processed, Parquet) → Glue Data Catalog → Athena → **SageMaker (este notebook)** → S3 (predictions).

Este notebook consome a base já tratada pelo Glue (`processed/vendas/`), formula o problema de ML,
remove atributos de vazamento de dado (*data leakage*), treina e avalia um modelo de regressão
contra uma referência simples, e gera previsões em lote para `raw/inferencia/dados_inferencia.csv`.

## 1. Configuração

In [ ]:
import sys, subprocess
for pkg in ["scikit-learn", "pyarrow"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import glob
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Nenhuma credencial e gravada aqui: o notebook usa a role de execucao do SageMaker (LabRole).
BUCKET = "bigdata-ml-varejo05-768366506781"
PROCESSED_PATH = f"s3://{BUCKET}/processed/vendas/"
INFERENCE_PATH = f"s3://{BUCKET}/raw/inferencia/dados_inferencia.csv"
PREDICTIONS_PATH = f"s3://{BUCKET}/predictions/previsoes.csv"

print("BUCKET:", BUCKET)
print("PROCESSED_PATH:", PROCESSED_PATH)
print("INFERENCE_PATH:", INFERENCE_PATH)
print("PREDICTIONS_PATH:", PREDICTIONS_PATH)

## 2. Carregamento dos dados processados (saída do Glue)

In [ ]:
df = pd.read_parquet(PROCESSED_PATH)
df["data"] = pd.to_datetime(df["data"])
df = df.sort_values(["loja", "produto", "data"]).reset_index(drop=True)
print("linhas:", df.shape[0], "| colunas:", df.shape[1])
print("periodo:", df["data"].min().date(), "a", df["data"].max().date())
print("valores ausentes por coluna:")
print(df.isna().sum())
df.head()

## 3. Formulação do problema

**Objetivo de negócio:** prever a demanda diária de produtos (`quantidade_vendida`) por loja e
produto, para apoiar decisões de reposição de estoque e planejamento de promoções.

**Variável alvo:** `quantidade_vendida` (unidades vendidas no dia, por loja/produto).

**Tipo de problema:** regressão com componente temporal (série cronológica dos dados deve ser
respeitada: não podemos treinar com dados do futuro para prever o passado; a divisão
treino/validação/teste segue a ordem cronológica, ver Seção 5).

**Atributos disponíveis antes da venda** (segundo o `LEIA-ME.md` do dataset):
- `preco`: preço planejado (já com desconto de promoção) — conhecido antes da venda.
- `promocao`: indicador de promoção planejada — conhecido antes da venda.
- `estoque`: unidades disponíveis na abertura do dia — conhecido antes da venda.
- `temperatura_prevista`: previsão meteorológica — conhecida antes da venda.
- `loja`, `produto`, `categoria`: atributos estruturais, sempre conhecidos.

**Atributo excluído por vazamento de dado (*data leakage*):** `receita_final` é apurada somente
ao **encerrar o dia** (é derivada da própria quantidade vendida), por isso não está disponível no
momento da previsão — e nem existe no arquivo de inferência. Não é usada como feature.

## 4. Engenharia de atributos

In [ ]:
# Atributos de calendario, conhecidos com antecedencia
df["mes"] = df["data"].dt.month
df["dia_semana"] = df["data"].dt.dayofweek
df["fim_de_semana"] = df["dia_semana"].isin([5, 6]).astype(int)

# Atributos de tendencia: media movel de vendas passadas por loja/produto.
# shift(1) garante que o dia atual nao entra na propria media (evita vazamento).
grp = df.groupby(["loja", "produto"])["quantidade_vendida"]
df["media_movel_7d"] = grp.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
df["media_movel_28d"] = grp.transform(lambda s: s.shift(1).rolling(28, min_periods=1).mean())

FEATURES_NUM = ["preco", "estoque", "temperatura_prevista", "media_movel_7d", "media_movel_28d"]
FEATURES_CAT = ["loja", "produto"]
FEATURES_BIN = ["promocao", "mes", "dia_semana", "fim_de_semana"]
X_COLS = FEATURES_NUM + FEATURES_CAT + FEATURES_BIN
TARGET = "quantidade_vendida"

print("features usadas:", X_COLS)
print("\nObs.: 'categoria' NAO entra como feature — e funcao deterministica de 'produto'")
print("(cada produto pertence a exatamente uma categoria), entao seria redundante/colinear.")

## 5. Divisão temporal (treino / validação / teste)

In [ ]:
train = df[(df["data"] >= "2024-01-01") & (df["data"] <= "2025-12-31")].copy()
val = df[(df["data"] >= "2026-01-01") & (df["data"] <= "2026-06-30")].copy()
test = df[(df["data"] >= "2026-07-01") & (df["data"] <= "2026-12-31")].copy()
print(f"treino: {len(train)} linhas ({train['data'].min().date()} a {train['data'].max().date()})")
print(f"validacao: {len(val)} linhas ({val['data'].min().date()} a {val['data'].max().date()})")
print(f"teste: {len(test)} linhas ({test['data'].min().date()} a {test['data'].max().date()})")

# Estatisticas de imputacao (moda de 'promocao') calculadas SOMENTE no treino, como exigido.
promo_mode = train["promocao"].mode()[0]
for part in (train, val, test):
    part["promocao"] = part["promocao"].fillna(promo_mode)
print("\nmoda de 'promocao' no treino (usada para imputar ausentes em val/teste/inferencia):", promo_mode)

## 6. Previsão de referência (baseline) e treinamento do modelo

In [ ]:
# Baseline simples: media historica de vendas por loja/produto, calculada so no treino.
baseline_map = train.groupby(["loja", "produto"])[TARGET].mean()
global_mean = train[TARGET].mean()

def baseline_predict(part):
    keys = list(zip(part["loja"], part["produto"]))
    return np.array([baseline_map.get(k, global_mean) for k in keys])

# Modelo: RandomForestRegressor. Imputacao (mediana) e one-hot sao ajustados (fit) so no treino,
# dentro do Pipeline — garantindo que nenhuma estatistica de validacao/teste "vaze" para o treino.
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
], remainder="passthrough")

model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)),
])

model.fit(train[X_COLS], train[TARGET])
print("modelo treinado.")

## 7. Avaliação: MAE e RMSE (modelo vs. referência)

In [ ]:
resultados = []
for nome, part in [("validacao", val), ("teste", test)]:
    y = part[TARGET]
    pred_modelo = model.predict(part[X_COLS])
    pred_base = baseline_predict(part)
    resultados.append({
        "conjunto": nome,
        "baseline_MAE": mean_absolute_error(y, pred_base),
        "baseline_RMSE": mean_squared_error(y, pred_base) ** 0.5,
        "modelo_MAE": mean_absolute_error(y, pred_modelo),
        "modelo_RMSE": mean_squared_error(y, pred_modelo) ** 0.5,
    })

resultados_df = pd.DataFrame(resultados)
resultados_df["reducao_MAE_%"] = 100 * (1 - resultados_df["modelo_MAE"] / resultados_df["baseline_MAE"])
resultados_df["reducao_RMSE_%"] = 100 * (1 - resultados_df["modelo_RMSE"] / resultados_df["baseline_RMSE"])
resultados_df

**Interpretação:** o RandomForest reduz o MAE em relação à referência (média histórica por
loja/produto) tanto na validação quanto no teste — ou seja, os atributos de preço, estoque,
promoção, calendário e tendência recente (`media_movel_7d/28d`) carregam sinal real sobre a
demanda além da média histórica simples. A métrica no conjunto de teste (jul–dez/2026, nunca
visto durante o ajuste do modelo nem na escolha de hiperparâmetros) é o número que deve ser
reportado como desempenho esperado do modelo em produção.

In [ ]:
importancias = pd.Series(
    model.named_steps["rf"].feature_importances_,
    index=model.named_steps["prep"].get_feature_names_out(),
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importancias.head(15)[::-1].plot(kind="barh")
plt.title("Importância dos atributos (RandomForest)")
plt.tight_layout()
plt.show()

## 8. Inferência: `dados_inferencia.csv` → `previsoes.csv`

Aplicamos a **mesma limpeza feita no Glue** (parsing de data em dois formatos, sentinelas -99/9999
→ nulo, padronização de `promocao`) e reaproveitamos apenas estatísticas aprendidas no treino
(moda de `promocao`, mediana das features numéricas embutida no pipeline).

In [ ]:
inf_raw = pd.read_csv(INFERENCE_PATH, dtype=str)

inf = inf_raw.copy()
inf["data"] = pd.to_datetime(inf["data"], format="ISO8601", errors="coerce")
mask_bad = inf["data"].isna()
inf.loc[mask_bad, "data"] = pd.to_datetime(inf_raw.loc[mask_bad, "data"], format="%d/%m/%Y", errors="coerce")

inf["preco"] = pd.to_numeric(inf["preco"], errors="coerce")
inf.loc[inf["preco"] == -99, "preco"] = np.nan
inf["estoque"] = pd.to_numeric(inf["estoque"], errors="coerce")
inf.loc[inf["estoque"] == 9999, "estoque"] = np.nan
inf["temperatura_prevista"] = pd.to_numeric(inf["temperatura_prevista"], errors="coerce")

promo_norm = inf["promocao"].str.strip().str.lower()
inf["promocao"] = promo_norm.map({"1": 1, "sim": 1, "0": 0, "nao": 0}).astype("float")
inf["promocao"] = inf["promocao"].fillna(promo_mode)

inf["mes"] = inf["data"].dt.month
inf["dia_semana"] = inf["data"].dt.dayofweek
inf["fim_de_semana"] = inf["dia_semana"].isin([5, 6]).astype(int)

# Media movel: ultimos 7/28 dias de historico REAL (ate 2026-12-31) por loja/produto.
# Limitacao: como a inferencia e para jan/2027 (apos o fim do historico), usamos o ultimo
# valor conhecido como "foto" fixa para todo o mes — nao simulamos a demanda dia a dia.
ultimo_hist = df.sort_values("data").groupby(["loja", "produto"]).tail(28)
mm7 = ultimo_hist.groupby(["loja", "produto"]).apply(lambda g: g.tail(7)["quantidade_vendida"].mean(), include_groups=False)
mm28 = ultimo_hist.groupby(["loja", "produto"])["quantidade_vendida"].mean()
inf["media_movel_7d"] = inf.set_index(["loja", "produto"]).index.map(mm7)
inf["media_movel_28d"] = inf.set_index(["loja", "produto"]).index.map(mm28)

pred = model.predict(inf[X_COLS])
pred = np.clip(pred, 0, None)  # quantidades nao podem ser negativas

previsoes = inf[["dataset_id", "data", "loja", "produto"]].copy()
previsoes["data"] = previsoes["data"].dt.strftime("%Y-%m-%d")
previsoes["quantidade_prevista"] = np.round(pred, 2)

print("linhas:", previsoes.shape[0])
print("chaves duplicadas:", previsoes.duplicated(subset=["dataset_id", "data", "loja", "produto"]).sum())
print("valores negativos:", (previsoes["quantidade_prevista"] < 0).sum())
previsoes.describe()

## 9. Salvar `previsoes.csv` no S3

In [ ]:
local_path = "/tmp/previsoes.csv"
previsoes.to_csv(local_path, index=False)

s3 = boto3.client("s3")
s3.upload_file(local_path, BUCKET, "predictions/previsoes.csv")
print("salvo em", PREDICTIONS_PATH)

previsoes.head(10)

## 10. Limitações da solução

- As features de tendência (`media_movel_7d/28d`) usam o último histórico real conhecido como um
  valor fixo para todo o período de inferência (jan/2027), em vez de recalcular dia a dia à medida
  que "novas vendas" ocorreriam — simplificação razoável para um protótipo, mas que tende a
  suavizar tendências dentro do próprio mês de inferência.
- `categoria` foi deliberadamente excluída do modelo por ser função determinística de `produto`
  (colinearidade); isso significa que o modelo não generaliza para produtos nunca vistos no
  treino — aceitável aqui pois o catálogo de produtos é fixo (10 SKUs).
- Valores ausentes de `preco`/`estoque`/`temperatura_prevista` são imputados pela mediana do
  treino (imputação simples); um valor ausente pode, em alguns casos, carregar informação (ex.:
  falha de sensor de estoque) que a mediana não captura.
- O modelo não foi publicado como endpoint do SageMaker (não exigido pelo enunciado); a
  inferência é em lote, executada diretamente neste notebook.